# SHAPE — Self-Healing Autonomous Programming Engine
## Notebook 1 of 4 · Introduction & Architecture

**Group 3 | ITAI2376-14562**  
Cherluk Sumdin · Olubukola Kukoyi · Taki Eddine Boubekri · Michael Rivas

---

**Project option:** Option 2 — Automation Agent  
**What this notebook covers:** Purpose, problem statement, software architecture, data flow,
package structure, and agent lifecycle. No agent is initialized here — that is Notebook 02.

**One shared package:** All four notebooks import from the same `shape/` package.
There is one `SHAPEAgent`, one `parse_llm_output`, one `InputGuard`, one `FeedbackMemory`.


## 1. Problem Statement

Every popular AI coding assistant (GitHub Copilot, ChatGPT, Claude) requires:
- A live internet connection
- A paid API subscription ($10–$60/month)
- Sending your code to a remote server

This makes them **completely unavailable** in real-world offline environments:
air-gapped hospital networks, remote field sites with no cell signal,
embedded robotics kits, and secure compartments.

**SHAPE's answer:** put the assistant on the device. Run the full agent locally
— on a laptop, a Raspberry Pi, or any Linux machine.
No cloud. No subscription. No data leaving the device.


## 2. Project Option: Option 2 — Automation Agent

| Requirement | SHAPE implementation | File |
|---|---|---|
| Task decomposition | ReAct loop breaks any coding request into Thought/Action/Observation steps | `shape/agent.py` |
| Sequential execution with error handling | Linter pre-screens, executor runs, agent retries on failure | `shape/tools/` |
| Progress monitoring | Verbose trace prints each step; RL stats via `agent.feedback.summary()` | `shape/rewards.py` |
| Adaptation over time | RL feedback loop adjusts system prompt between sessions | `shape/rewards.py` |
| Use ≥ 2 external tools | 4 tools: code_executor, linter, file_manager, doc_search | `shape/tools/` |
| Input validation | InputGuard regex screen (pre-LLM, < 1 ms) | `shape/safety.py` |


## 3. Software Architecture

```
User Input (text)
        │
        ▼
┌───────────────────┐
│   InputGuard      │  ← Pre-LLM safety screen (regex, < 1 ms)
│   shape/safety.py │    Blocks harmful/out-of-scope requests
└────────┬──────────┘
         │
         ▼
┌─────────────────────────────────────────────────────────────┐
│  SHAPEAgent.run()  — shape/agent.py                        │
│                                                             │
│  SystemMessage = REACT_SYSTEM + RL policy hint             │
│  HumanMessage  = trimmed history + user input              │
│                                                             │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  ReAct Loop (max 8 iterations)                      │   │
│  │                                                     │   │
│  │  LLM output → parse_llm_output()                   │   │
│  │       │                                             │   │
│  │       ├── Action: linter        → pyflakes          │   │
│  │       ├── Action: code_executor → subprocess        │   │
│  │       ├── Action: file_manager  → workspace I/O     │   │
│  │       ├── Action: doc_search    → FAISS / keyword   │   │
│  │       └── Final Answer          → return to user    │   │
│  └─────────────────────────────────────────────────────┘   │
└────────┬────────────────────────────────────────────────────┘
         │
         ▼
┌───────────────────┐
│  FeedbackMemory   │  ← User rates 1–5 → rolling avg → policy hint
│  shape/rewards.py │    Hint injected into next SystemMessage
└───────────────────┘
```


## 4. Numbered Data Flow: User Request → Final Answer

| Step | Location | Action | Code |
|---|---|---|---|
| 1 | `main.py` | User types a coding request at the SHAPE> prompt | `main.py` |
| 2 | `shape.safety` | InputGuard screens for blocked patterns (pre-LLM, < 1 ms) | `safety.py:InputGuard.check()` |
| 3 | `shape.agent` | RL hint retrieved and appended to system prompt | `rewards.py:get_policy_hint()` |
| 4 | `shape.agent` | History trimmed to memory window, messages assembled | `memory.py:trimmed_messages()` |
| 5 | `shape.agent` | LLM invoked (CodeLlama 7B via Ollama, offline) | `agent.py:_step()` |
| 6 | `shape.parser` | Output parsed: Final Answer / Action / Fallback | `parser.py:parse_llm_output()` |
| 7 | `shape.tools` | Tool called (linter → executor → file_manager → doc_search) | `tools/` |
| 8 | `shape.agent` | Tool output fed back as Observation; loop repeats from step 5 | `agent.py:run()` |
| 9 | `main.py` | Final Answer returned and printed | `main.py` |
| 10 | `shape.rewards` | User rates 1–5; score recorded; rolling avg updates policy hint | `rewards.py:FeedbackMemory.record()` |

**Alert reliability:** Step 2 fires before the LLM. The decision to block a request is always
rule-based regex — the LLM is never called for blocked inputs.


## 5. Package Structure — One Codebase, Four Notebooks

```
shape/
├── __init__.py        public API exports
├── agent.py           SHAPEAgent + build_agent() — the only constructor
├── parser.py          parse_llm_output() — Notebook 02
├── memory.py          make_history(), trimmed_messages() — Notebook 02
├── prompts.py         REACT_SYSTEM template — Notebook 02
├── rewards.py         FeedbackMemory — Notebook 04
├── safety.py          InputGuard — Notebook 03
├── sensors.py         SensorReading, parse_csv_line() — Notebook 03
└── tools/
    ├── __init__.py    get_tools(config)
    ├── code_executor.py
    ├── linter.py
    ├── file_manager.py
    └── doc_search.py

tests/
├── test_parser.py    9 tests — all branches
├── test_agent.py     12 tests — safety, memory, rewards
└── test_tools.py     11 tests — all error paths
```

Every notebook does `from shape.xxx import yyy`.
No notebook reimplements any logic.


In [ ]:
# ── Colab / environment setup ─────────────────────────────────────────────────
# Run once per session. Skip if already in SHAPE_Core directory.
import os, sys, subprocess
from pathlib import Path

if not Path('shape').exists():
    # Colab: clone repo
    subprocess.run(['git', 'clone',
                    'https://github.com/mrivassnj-svg/SHAPE_Core.git'], check=True)
    os.chdir('SHAPE_Core')

# Always install with the same Python the kernel is using (sys.executable).
# This guarantees pyflakes, langchain, etc. are on the kernel's sys.path.
subprocess.run([sys.executable, '-m', 'pip', 'install',
                '-r', 'requirements.txt', '--quiet'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'langchain-ollama==0.1.3', '--quiet', '--force-reinstall'], check=True)

# Always ensure the package root is on sys.path
root = Path('.').resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print('Working directory:', os.getcwd())
print('shape/ found:', Path('shape').exists())
print('Python:', sys.executable)

In [ ]:
# Verify the package structure — all modules must be importable from shape/

from shape          import __version__, __authors__
from shape.parser   import parse_llm_output
from shape.memory   import make_history, trimmed_messages
from shape.prompts  import REACT_SYSTEM
from shape.rewards  import FeedbackMemory, MIN_RECORDS_FOR_HINT
from shape.safety   import InputGuard, _BLOCKED_PATTERNS, _OUT_OF_SCOPE_PATTERNS
from shape.tools    import get_tools
from shape.agent    import build_agent, SHAPEAgent

print(f'SHAPE v{__version__}')
print(f'Authors: {", ".join(__authors__)}')
print()
print('Module inventory:')
print(f'  shape.parser   — parse_llm_output()    ← Notebook 02')
print(f'  shape.memory   — make_history()         ← Notebook 02')
print(f'  shape.prompts  — REACT_SYSTEM           ← Notebook 02')
print(f'  shape.rewards  — FeedbackMemory         ← Notebook 04')
print(f'  shape.safety   — InputGuard ({len(_BLOCKED_PATTERNS)} blocked, {len(_OUT_OF_SCOPE_PATTERNS)} out-of-scope patterns)  ← Notebook 03')
print(f'  shape.tools    — get_tools()            ← Notebook 03')
print(f'  shape.agent    — build_agent()          ← Notebooks 02–04')

In [ ]:
# Run the test suite — 32 tests covering parser, safety, memory, tools.
# All must pass before proceeding to Notebook 02.

import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short', '-q'],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
    raise RuntimeError('Tests failed — do not proceed until all pass.')
print('All tests passed. ✓')

In [ ]:
# Interactive ReAct Visualizer
# Teaching mode = deterministic scenarios (no LLM)
# Live Trace mode = observes real agent.run() output

from pathlib import Path
from IPython.display import display, HTML, IFrame

viz = Path('notebooks/shape_react_visualizer.html')
if not viz.exists():
    viz = Path('shape_react_visualizer.html')

if viz.exists():
    try:
        import google.colab  # noqa
        display(HTML(viz.read_text()))
    except ImportError:
        display(IFrame(src=str(viz), width='100%', height='820px'))
    print('Visualizer loaded.')
    print('  Teaching mode: deterministic — safe for grading/demo')
    print('  Live Trace:    connects to actual agent.run() in Notebook 02')
else:
    print('Open notebooks/shape_react_visualizer.html in a browser.')

## Summary

| Item | Demonstrated |
|---|---|
| Project option | Option 2 — Automation Agent, all 6 requirements mapped |
| Software architecture | InputGuard → ReAct loop → 4 tools → RL feedback |
| Data flow | 10-step numbered sequence from user input to rated response |
| Safety | Pre-LLM guard: blocked inputs never reach the model |
| Package structure | 9 modules, one import path per notebook |
| Tests | 32 unit tests — all pass before proceeding |

▶ **Continue to `02_SHAPE_ReAct_Engine.ipynb`**
